# Weeks 5 and 6: baselines for next hour's energy, and how far to trust them

Run the cells in order. Part A scores the plant's rule, Part B is the by-hand table, Part C is the regression, Part E measures how sure you can be, and Part F turns that into a booking. Parts B and D send you to the terminal to make tests pass before the cells that follow will run.

Nothing here edits your data file.

In [17]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
from paths import DATA_RAW, ARTIFACTS, ensure_output_dirs
from baselines import *
from uncertainty import *

STUDENT = 'muhammadasad-77'
log = load_log(DATA_RAW / f'shift_week_{STUDENT}.csv')
log = add_target(log)
log['next_hour_night'] = next_hour_is_night(log)
train, test = split_by_day(log, test_days=2)
print(len(train), 'training hours,', len(test), 'test hours')
test.head()

414 training hours, 87 test hours


,timestamp,line_id,shift,units_produced,downtime_min,energy_kwh,rejected_units,motor_temp_c,energy_kwh_smoothed,hour,energy_next,next_hour_night
138,2026-01-18 00:00:00,Line_A,Night,399,4.2,154.4,4,73.9,160.1,0,165.8,1
139,2026-01-18 01:00:00,Line_A,Night,454,5.5,165.8,5,75.1,161.5,1,157.2,1
140,2026-01-18 02:00:00,Line_A,Night,385,2.3,157.2,4,74.1,158.0,2,158.8,1
141,2026-01-18 03:00:00,Line_A,Night,387,3.7,158.8,3,75.1,157.9,3,157.1,1
142,2026-01-18 04:00:00,Line_A,Night,404,1.0,157.1,4,77.9,156.8,4,156.6,1


## Part A: score the incumbent rule

The plant's rule is *next hour looks like this one*. Score it first and write the number down. This is the bar.

The average miss is worked out in the open here: the size of each miss, averaged. In Part B you turn that arithmetic into the function `mean_absolute_error`, which also has to cope with missing values.

In [18]:
rule = naive_last_hour(test)
miss = (test[TARGET] - rule).abs()     # the arithmetic in the open; Part B makes it a function
print('Incumbent rule, average miss (kWh):', round(miss.mean(), 2))
print('Hours short:', hours_short(test[TARGET], rule), 'of', len(test))

Incumbent rule, average miss (kWh): 5.45
Hours short: 45 of 87


## Part B: three baselines by hand, then in code

Six consecutive hours of one line, from your training days. For each row write down, **with a pencil**, what each baseline would predict for the *next* hour, then the miss. Average the misses.

| Hour | Energy this hour | Actual next hour | Naive (last hour) | Line mean | Same hour yesterday |
|---|---:|---:|---:|---:|---:|
| | | | | | |
| | | | | | |
| | | | | | |
| | | | | | |
| | | | | | |
| | | | | | |
| **Average miss** | | | | | |

Then, in a terminal at the project root, write the three TODO functions in `src/baselines.py` until `pytest -q tests/test_baselines.py` is green, and run the check cell below.

In [19]:
six = train[train.line_id == 'Line_A'].iloc[30:36]
six[['timestamp', 'energy_kwh', TARGET]]

,timestamp,energy_kwh,energy_next
30,2026-01-13 12:00:00,164.2,159.2
31,2026-01-13 13:00:00,159.2,172.9
32,2026-01-13 14:00:00,172.9,161.3
33,2026-01-13 15:00:00,161.3,161.5
34,2026-01-13 16:00:00,161.5,164.6
35,2026-01-13 17:00:00,164.6,150.2


In [20]:
# Check your pencil work. Runs once the Part B functions are written.
checks = pd.DataFrame({
    'actual_next': six[TARGET],
    'naive': naive_last_hour(six),
    'line_mean': line_mean(train, six),
    'same_hour_yesterday': same_hour_yesterday(train).loc[six.index],
})
checks['miss_naive'] = (checks.actual_next - checks.naive).abs()
checks

,actual_next,naive,line_mean,same_hour_yesterday,miss_naive
30,159.2,164.2,160.169565,171.3,5.0
31,172.9,159.2,160.169565,169.4,13.7
32,161.3,172.9,160.169565,159.1,11.6
33,161.5,161.3,160.169565,163.3,0.2
34,164.6,161.5,160.169565,164.5,3.1
35,150.2,164.6,160.169565,142.5,14.4


## Part C: regression with features that exist at the top of the hour

The cell below adds two engineered features (it needs `same_hour_yesterday` from Part B). Every candidate is then a column in your frame. For each, ask: at the top of the hour, is this known? `passes_top_of_hour_test` gives the course's answer; the argument is yours to write in the interpretation note.

Your file has one more column than is listed here. It is not labelled, and it will make a regression look superb. Find it, say how you knew, and do not use it.

In [21]:
log = add_candidate_features(log)
train, test = split_by_day(log, test_days=2)
candidates = ['energy_kwh', 'units_produced', 'downtime_min', 'motor_temp_c', 'hour',
              'next_hour_night', 'energy_last_hour', 'energy_same_hour_yesterday']
for c in candidates:
    print(f'{c:28s}', 'known at top of hour' if passes_top_of_hour_test(c) else 'NOT known: do not use')

energy_kwh                   known at top of hour
units_produced               known at top of hour
downtime_min                 known at top of hour
motor_temp_c                 known at top of hour
hour                         known at top of hour
next_hour_night              known at top of hour
energy_last_hour             known at top of hour
energy_same_hour_yesterday   known at top of hour


In [22]:
FEATURES = ['energy_kwh', 'next_hour_night']   # <-- your choice; two or three, all passing the test
coef = fit_regression(train[FEATURES], train[TARGET])
print(coef.round(3))
regression = predict(test[FEATURES], coef)

intercept          27.599
energy_kwh          0.833
next_hour_night    -3.659
dtype: float64


In [23]:
predictions = {
    'naive_last_hour (the rule)': naive_last_hour(test),
    'line_mean': line_mean(train, test),
    'same_hour_yesterday': same_hour_yesterday(log).loc[test.index],
    'regression': regression,
}
card = scorecard(test, predictions)
ensure_output_dirs()
card.to_csv(ARTIFACTS / 'scorecard.csv')
card

,mean_absolute_error,hours_short,hours_scored
model,,,
naive_last_hour (the rule),5.45,45,87
line_mean,6.50,42,87
same_hour_yesterday,6.50,56,87
regression,5.28,50,87


### The hour every model missed

Find the test hour with the largest miss for the rule. You have met it before.

In [24]:
resid = (test[TARGET] - predictions['naive_last_hour (the rule)']).abs()
worst = test.loc[resid.idxmax(), ['timestamp', 'line_id', 'motor_temp_c', 'energy_kwh', TARGET]]
print('Largest miss for the rule:', round(resid.max(), 1), 'kWh')
worst

Largest miss for the rule: 48.1 kWh


timestamp       2026-01-18 22:00:00
line_id                      Line_C
motor_temp_c                  112.0
energy_kwh                    204.8
energy_next                   156.7
Name: 494, dtype: object

## Part D: make the uncertainty tests pass

In a terminal at the project root: `pytest -q`. Six tests fail until `residual_spread`, `lag1_autocorrelation` and `margin_for_price` in `src/uncertainty.py` are written. Come back when all 26 are green; Parts E and F use all three.

## Part E: how sure can you be?

First, add **one** feature to your Part C regression. It must pass the top-of-the-hour test, and it must not already be in `FEATURES`. A legitimate feature does not have to help; if yours does not, say so in the note and try another.

Each model is stored with its training predictions, its test predictions, and how many numbers it fitted. The rule fits none; the line mean fits one per line; a regression fits one per coefficient, intercept included.

In [25]:
NEW_FEATURE = 'energy_last_hour'
assert NEW_FEATURE in candidates and NEW_FEATURE not in FEATURES, 'choose one new feature from the candidates list'

FEATURES_IMPROVED = FEATURES + [NEW_FEATURE]
coef_improved = fit_regression(train[FEATURES_IMPROVED], train[TARGET])
print(coef_improved.round(3))

models = {
    'rule': (naive_last_hour(train), naive_last_hour(test), 0),
    'line mean': (line_mean(train, train), line_mean(train, test), train['line_id'].nunique()),
    'regression': (predict(train[FEATURES], coef), predict(test[FEATURES], coef), len(coef)),
    'improved': (predict(train[FEATURES_IMPROVED], coef_improved), predict(test[FEATURES_IMPROVED], coef_improved), len(coef_improved)),
}
scorecard(test, {name: m[1] for name, m in models.items()})

intercept           24.114
energy_kwh           0.630
next_hour_night     -3.865
energy_last_hour     0.226
dtype: float64


,mean_absolute_error,hours_short,hours_scored
model,,,
rule,5.45,45,87
line mean,6.50,42,87
regression,5.28,50,87
improved,4.96,49,87


### Three measurements, all on the training hours, then one check on the test hours

- **$s$**, the typical size of a miss. With residuals $e_i = y_i - \hat y_i$, $n$ of them, and $p$ numbers fitted, $$s = \sqrt{\frac{\sum_i e_i^2}{n - p}}.$$
- **Streaks**: the lag-1 autocorrelation $\rho_1 = \operatorname{Corr}(e_t, e_{t-1})$, the correlation between each hour's miss and the previous hour's, within a line. Near 0, each hour is close to independent evidence. Clearly positive, misses come in runs, and any interval that treats the hours as independent is too narrow.
- **Coverage**: $\hat y \pm 1.96\,s$ is a nominal range built on a normal reference. It would hold about 95% of hours if the misses were independent, normal, equally spread throughout, and $s$ were known exactly; with hundreds of hours the lecture's $t_{0.975,5} = 2.571$ for six values is close to 1.96. The last column is its **empirical coverage**: the share of test hours that actually fell inside. Treat it as a diagnostic of the range. It does not make the range a calibrated 95% prediction interval, and streaks, skewed misses or a spread that changes can all move it.

In [26]:
rows = []
for name, (fit_train, fit_test, n_params) in models.items():
    r_train = residuals(train[TARGET], fit_train)
    s = residual_spread(r_train, n_params)
    rows.append({'model': name,
                 'numbers_fitted': n_params,
                 's_kwh': round(s, 2),
                 'lag1_autocorrelation': round(lag1_autocorrelation(r_train, train['line_id']), 2),
                 'test_share_within_1.96s': round(share_within(residuals(test[TARGET], fit_test), 1.96 * s), 2)})
spread = pd.DataFrame(rows).set_index('model')
spread.to_csv(ARTIFACTS / 'spread.csv')
spread

,numbers_fitted,s_kwh,lag1_autocorrelation,test_share_within_1.96s
model,,,,
rule,0,6.25,-0.25,0.94
line mean,3,7.72,0.67,0.95
regression,3,5.85,-0.13,0.90
improved,4,5.71,0.04,0.92


### One assumption you can check: the same spread before day hours and night hours

A single $s$ assumes every hour is equally hard to predict. Split the improved model's training misses by whether the hour being predicted is a night hour. (The model's numbers were fitted on all hours together, so within one group nothing is subtracted for them; with two hundred hours a group, that changes $s$ by about one per cent.)

In [27]:
r_train = residuals(train[TARGET], models['improved'][0])
for flag, label in [(0, 'predicting a day hour'), (1, 'predicting a night hour')]:
    m = train['next_hour_night'] == flag
    print(f'{label:24s} hours: {int(r_train[m].notna().sum()):3d}   s = {residual_spread(r_train[m], 0):.2f} kWh')

predicting a day hour    hours: 210   s = 5.65 kWh
predicting a night hour  hours: 201   s = 5.72 kWh


### How firmly is each coefficient pinned down?

Refit the improved model once for each training day, leaving that day out. The lowest and highest value each coefficient takes across the refits is its **leave-one-day-out sensitivity range**: how much the number depends on which days happened to be in the training data. It is a sensitivity check and makes no probability claim, so it is not the plant's version of the lecture's 95% confidence interval of 44 to 52; call it by its own name in the note. Whole days are left out together so that hours of the same day, which may share a streak of misses, stay together.

In [28]:
refits = refit_leaving_out_each_day(train[FEATURES_IMPROVED], train[TARGET], train['timestamp'])
print(refits.round(3), end='\n\n')
pd.DataFrame({'all training days': coef_improved, 'lowest': refits.min(), 'highest': refits.max()}).round(3)

              intercept  energy_kwh  next_hour_night  energy_last_hour
day_left_out                                                          
2026-01-12       22.684       0.644           -3.823             0.221
2026-01-13       24.451       0.651           -3.938             0.203
2026-01-14       25.176       0.620           -3.818             0.228
2026-01-15       23.868       0.609           -3.835             0.248
2026-01-16       24.124       0.625           -3.803             0.231
2026-01-17       24.506       0.630           -3.974             0.223



,all training days,lowest,highest
intercept,24.114,22.684,25.176
energy_kwh,0.630,0.609,0.651
next_hour_night,-3.865,-3.974,-3.803
energy_last_hour,0.226,0.203,0.248


### The hour every model missed, against the range

The largest test miss, against the nominal range $\hat y \pm 1.96\,s$. Look at the second row, the hour being predicted, and its motor temperature. Name it and say who could explain it; diagnosing misses in detail is Week 7.

In [14]:
r_test = residuals(test[TARGET], models['improved'][1])
worst = r_test.abs().idxmax()
print('Largest test miss:', round(r_test[worst], 1), 'kWh; nominal range y_hat +/-',
      round(1.96 * spread.loc['improved', 's_kwh'], 1), 'kWh')
# The row the prediction was made from, and the row after it: the hour being predicted.
log.loc[worst:worst + 1, ['timestamp', 'line_id', 'motor_temp_c', 'energy_kwh', TARGET]]

Largest test miss: 49.3 kWh; nominal range y_hat +/- 11.2 kWh


,timestamp,line_id,motor_temp_c,energy_kwh,energy_next
493,2026-01-18 21:00:00,Line_C,76.0,157.1,204.8
494,2026-01-18 22:00:00,Line_C,112.0,204.8,156.7


## Part F: book with a margin

At the top of each hour the plant books next hour's energy. A kWh used beyond the booking costs `PRICE_SHORT`; a kWh booked and not used costs `PRICE_OVER`. In the lecture's notation $C_u = 3$ and $C_o = 1$, so the cheapest booking $q$ satisfies $$F(q) = \frac{C_u}{C_u + C_o} = 0.75:$$ it is short on about one hour in four. With $e = y - \hat y$ and $q = \hat y + m$, the margin $m$ is the empirical 75th percentile of the **training** residuals, which is what `margin_for_price` returns.

Every model gets its own margin, from its own training misses, and every booking is scored on the same test hours at the same prices. That way a booking is compared with a booking, which is the fair comparison.

In [29]:
print('A kWh short costs', PRICE_SHORT, 'x a kWh over; aim to be short on',
      PRICE_OVER / (PRICE_SHORT + PRICE_OVER), 'of hours')
bookings, margins = {}, {}
for name, (fit_train, fit_test, _) in models.items():
    margins[name] = margin_for_price(residuals(train[TARGET], fit_train))
    bookings[f'{name}, no margin'] = fit_test
    bookings[f'{name}, + {margins[name]:.1f} kWh'] = fit_test + margins[name]
book = booking_scorecard(test[TARGET], bookings)
book['share_of_hours_short'] = (book['hours_short'] / book['hours_scored']).round(2)
book.to_csv(ARTIFACTS / 'booking_scorecard.csv')
book.sort_values('cost')

A kWh short costs 3.0 x a kWh over; aim to be short on 0.25 of hours


,kwh_over,kwh_short,hours_short,cost,hours_scored,share_of_hours_short
booking,,,,,,
"improved, + 3.6 kWh",356.8,118.4,23,712.1,87,0.26
"regression, + 3.7 kWh",381.5,127.2,26,763.1,87,0.30
"rule, + 4.0 kWh",447.8,115.1,23,793.0,87,0.26
"line mean, + 6.0 kWh",549.3,121.8,25,914.8,87,0.29
"improved, no margin",178.6,252.6,49,936.6,87,0.56
"rule, no margin",230.7,243.8,45,962.1,87,0.52
"regression, no margin",196.1,263.5,50,986.6,87,0.57
"line mean, no margin",233.3,332.2,42,1229.9,87,0.48


### The price moves the margin

The ratio is the plant's decision, and the answer depends on it. See how the improved model's margin changes.

In [16]:
r_train = residuals(train[TARGET], models['improved'][0])
for price_short in [1.0, 3.0, 10.0]:
    print(f'{price_short:4.0f} : 1  ->  margin {margin_for_price(r_train, price_short, 1.0):+.1f} kWh')

   1 : 1  ->  margin +0.0 kWh
   3 : 1  ->  margin +3.6 kWh
  10 : 1  ->  margin +7.6 kWh


## Part G: the two notes

Complete `docs/interpretation_note.md` (Parts A to C) and `docs/uncertainty_note.md` (Parts E and F). Six questions each, one sentence each, each backed by a number from `artifacts/scorecard.csv`, `artifacts/spread.csv`, `artifacts/booking_scorecard.csv` or a table above. Name each range for what it is: a coverage check, a sensitivity range, or a margin. Push before you leave; the notes may be finished after the session.